# 🎯 Bangla Diarization — Fine-tune Pyannote Segmentation Model

## What This Notebook Does (Plain English)

Your current pipeline uses pyannote's segmentation model which was trained on English/European speech.
This notebook **teaches it Bangla speech patterns** by continuing its training on YOUR labeled data.

### The Big Picture
```
BEFORE: pretrained model (English) --> your Bangla audio --> DER 0.32
AFTER:  finetuned model (Bangla)   --> your Bangla audio --> DER ~0.24-0.28 (expected)
```

### What gets trained?
Only the **segmentation model** (~6MB). This is the part that detects:
- Where speech starts/ends
- Where one speaker stops and another begins
- Where two speakers overlap

The embedding model (which identifies WHO is speaking) stays frozen — it's already good enough.

### What you need
- Your training audio files (WAVs)
- Your training annotation CSVs (start_time, end_time, speaker_id columns)
- A HuggingFace token (same one you already use)
- T4 GPU on Kaggle (~45 min to fine-tune)

### Where the saved model goes
At the end of this notebook, the finetuned model is saved to:
`/kaggle/working/finetuned_segmentation/best_model.ckpt`

You then attach this as a Kaggle dataset to your **inference notebook** and load it with 2 lines of code.

---

## Step 1: Install Dependencies

In [ ]:
# # Install dependencies — same versions as your inference notebook
# !apt-get install -y ffmpeg -q

# !pip install torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu118 -q
# !pip install "pyannote.audio==4.0.3" -q
# !pip install "pytorch-lightning>=2.0" -q
# !pip install "numpy<2.0" "pandas>=1.3" -q

# !apt-get install -y ffmpeg -q  # usually already available on Kaggle
# !pip install torchcodec==0.7.0
# !pip install "pyannote.audio==3.3.1" "scipy==1.13.1" -q

# print("✅ Dependencies installed")



using_community = 1
print(f"using_community = {using_community}")
if using_community == 1 : 
    !apt-get install -y ffmpeg
    # !pip install torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu118
    !pip install torchcodec==0.7.0
    !pip install "pyannote.audio==4.0.4"
    !pip install "numpy>=1.24" "pandas>=1.3"
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else :
    !apt-get install -y ffmpeg -q  # usually already available on Kaggle
    !pip install torchcodec==0.7.0
    !pip install "pyannote.audio==3.3.2" "scipy==1.13.1" -q


## Step 2: Verify Everything Works

In [ ]:
# import torch
# import torchaudio
# import pyannote.audio
# import pytorch_lightning

# print(f"torch:              {torch.__version__}")
# print(f"torchaudio:         {torchaudio.__version__}")
# print(f"pyannote.audio:     {pyannote.audio.__version__}")
# print(f"pytorch-lightning:  {pytorch_lightning.__version__}")
# print(f"CUDA available:     {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"GPU:                {torch.cuda.get_device_name(0)}")

# print("\n✅ All good!")

#checking versions
import torch
import torchaudio
import torchcodec
import numpy as np
import pandas as pd
import pyannote.audio

print("=" * 45)
print("   DEPENDENCY VERSION CHECK")
print("=" * 45)

versions = {
    "torch":          (torch.__version__,          "2.8.0"),
    "torchaudio":     (torchaudio.__version__,     "2.8.0"),
    "torchcodec":     (torchcodec.__version__,     "0.7.0"),
    "numpy":          (np.__version__,             ">=1.24"),
    "pandas":         (pd.__version__,             ">=1.3"),
    "pyannote.audio": (pyannote.audio.__version__, ">=4.0.0"),
}

all_good = True
for lib, (installed, expected) in versions.items():
    status = "✅" if installed else "❌"
    print(f"  {status}  {lib:<18} {installed:<12}  (expected: {expected})")
    if not installed:
        all_good = False

print("=" * 45)

# CUDA check
cuda_available = torch.cuda.is_available()
cuda_status = "✅" if cuda_available else "⚠️  (CPU only)"
print(f"  {'✅' if cuda_available else '⚠️ '}  {'CUDA':<18} {str(cuda_available):<12}")
if cuda_available:
    print(f"       {'GPU':<18} {torch.cuda.get_device_name(0)}")
    print(f"       {'CUDA version':<18} {torch.version.cuda}")

# ffmpeg check
import subprocess
try:
    result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
    ffmpeg_ver = result.stdout.split("\n")[0].split(" ")[2]
    print(f"  ✅  {'ffmpeg':<18} {ffmpeg_ver}")
except FileNotFoundError:
    print(f"  ❌  {'ffmpeg':<18} NOT FOUND  ← required for pyannote 4.x!")
    all_good = False

print("=" * 45)
print("  ✅ All good — ready to run!" if all_good else "  ❌ Fix the above before proceeding.")
print("=" * 45)

In [ ]:
import torch

# PyTorch 2.6+ security fix — run this ONCE at the start
if not hasattr(torch, '_load_patched'):
    _original_torch_load = torch.load
    
    def _trusted_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _original_torch_load(*args, **kwargs)
    
    torch.load = _trusted_load
    torch._load_patched = True
    print("✅ torch.load patched")
else:
    print("✅ torch.load already patched (skipping)")

## Step 3: Configuration

**You only need to change things in this cell.**

In [ ]:
import os

# ============================================================
# CONFIGURATION — edit these paths/values if needed
# ============================================================

HF_TOKEN = "your_token"  # your existing HF token

# Where your training data lives (same as tuning notebook)
TRAIN_AUDIO_DIR      = "/kaggle/input/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/audio"
TRAIN_ANNOTATION_DIR = "/kaggle/input/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/annotation"

# Where to save the finetuned model
OUTPUT_DIR = "/kaggle/working/finetuned_segmentation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training settings
# ----------------------------------------------------------------
# NUM_EPOCHS: How many times to go through ALL your training data.
#   - Start with 5. If DER keeps improving, try 10.
#   - More is NOT always better — the model can overfit.
NUM_EPOCHS = 30

# LEARNING_RATE: How fast the model changes its weights.
#   - 1e-4 is the standard safe value for fine-tuning.
#   - Don't change this unless you know what you're doing.
LEARNING_RATE = 1e-4

# BATCH_SIZE: How many audio chunks to process at once.
#   - 8 is safe for T4 (16GB VRAM). Lower if you get OOM errors.
BATCH_SIZE = 8

# CHUNK_DURATION: Length of audio chunks used for training (seconds).
#   - Pyannote processes audio in fixed-length windows.
#   - 10s matches the pretrained model's window size — don't change this.
CHUNK_DURATION = 10.0

# Max speakers expected in a single 10s chunk.
#   - Your data has up to 22 speakers total, but rarely more than 4 in 10s.
MAX_SPEAKERS_PER_CHUNK = 3

# ----------------------------------------------------------------
print("✅ Configuration loaded")
print(f"   Training audio:       {TRAIN_AUDIO_DIR}")
print(f"   Training annotations: {TRAIN_ANNOTATION_DIR}")
print(f"   Output dir:           {OUTPUT_DIR}")
print(f"   Epochs:               {NUM_EPOCHS}")
print(f"   Learning rate:        {LEARNING_RATE}")
print(f"   Batch size:           {BATCH_SIZE}")

## Step 4: Prepare Data

### What's a pyannote "Protocol"?
Pyannote doesn't read files directly. It uses a **protocol** — basically a structured way
to tell it: "here are your training files, here are the annotations, here are the test files."

We build this protocol programmatically from your CSV annotations.

### What's an RTTM file?
Pyannote expects annotations in **RTTM format** (Rich Transcription Time Mark).
Each line looks like:
```
SPEAKER filename 1 start_time duration <NA> <NA> speaker_id <NA> <NA>
```
We convert your CSVs to this format automatically below.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

def to_seconds(val) -> float:
    """Convert HH:MM:SS or float string to seconds."""
    val = str(val).strip()
    if ':' in val:
        parts = val.split(':')
        if len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
        elif len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
    return float(val)


def csv_to_rttm(csv_path: str, file_id: str) -> str:
    """
    Convert your annotation CSV to RTTM format.
    - Strips overlapping segments (first-speaker-wins, matching judge's rule)
    - Cleans speaker IDs from float strings (1.0 -> 1)
    
    RTTM format per line:
    SPEAKER <file_id> 1 <start> <duration> <NA> <NA> <speaker> <NA> <NA>
    """
    df = pd.read_csv(csv_path)
    cols = df.columns.tolist()
    
    # Auto-detect column names
    start_col   = next((c for c in cols if 'start'   in c.lower()), None)
    end_col     = next((c for c in cols if 'end'     in c.lower()), None)
    speaker_col = next((c for c in cols if 'speaker' in c.lower() or 'label' in c.lower()), None)
    
    if not all([start_col, end_col, speaker_col]):
        raise ValueError(f"Cannot find start/end/speaker columns in {csv_path}. Columns: {cols}")
    
    # Convert timestamps
    df[start_col] = df[start_col].apply(to_seconds)
    df[end_col]   = df[end_col].apply(to_seconds)
    df = df.dropna(subset=[start_col, end_col])
    
    # # Sort by start time (required before overlap removal)
    # df = df.sort_values(start_col).reset_index(drop=True)
    
    # # Remove overlapping segments — keep first speaker only
    # # This mirrors the judge's rule: speaker who started first wins
    # filtered_rows = []
    # last_end = 0.0
    # for _, row in df.iterrows():
    #     start = float(row[start_col])
    #     end   = float(row[end_col])
    #     if start >= last_end:       # no overlap with previous segment
    #         filtered_rows.append(row)
    #         last_end = end
    #     # else: overlapping segment — skip it entirely
    
    # df = pd.DataFrame(filtered_rows)
    
    # Build RTTM lines
    lines = []
    for _, row in df.iterrows():
        start    = float(row[start_col])
        end      = float(row[end_col])
        duration = end - start
        if duration <= 0:
            continue
        # Skip rows with missing speaker label
        if pd.isna(row[speaker_col]):
            continue
        # FIX: cast float speaker IDs (1.0, 2.0) to clean int strings ("1", "2")
        speaker = str(int(float(str(row[speaker_col]).strip())))
        lines.append(
            f"SPEAKER {file_id} 1 {start:.3f} {duration:.3f} <NA> <NA> {speaker} <NA> <NA>"
        )
    
    return "\n".join(lines)


# ── Convert all CSVs to RTTM ──────────────────────────────────────
rttm_dir = os.path.join(OUTPUT_DIR, "rttm")
os.makedirs(rttm_dir, exist_ok=True)

audio_files      = sorted(Path(TRAIN_AUDIO_DIR).glob("*.wav"))
annotation_files = sorted(Path(TRAIN_ANNOTATION_DIR).glob("*.csv"))

# Match audio files to their annotations by stem name
pairs = []
for audio_path in audio_files:
    ann_path = Path(TRAIN_ANNOTATION_DIR) / f"{audio_path.stem}.csv"
    if ann_path.exists():
        pairs.append((audio_path, ann_path))
    else:
        print(f"  ⚠️  No annotation found for {audio_path.stem}, skipping")

print(f"Found {len(pairs)} audio+annotation pairs")

# Convert to RTTM files
rttm_lines_all = []
for audio_path, ann_path in pairs:
    file_id  = audio_path.stem
    rttm_str = csv_to_rttm(str(ann_path), file_id)
    
    # Save individual RTTM file
    rttm_path = os.path.join(rttm_dir, f"{file_id}.rttm")
    with open(rttm_path, "w") as f:
        f.write(rttm_str)
    
    rttm_lines_all.append(rttm_str)

# Save combined RTTM (all files in one)
combined_rttm_path = os.path.join(OUTPUT_DIR, "train.rttm")
with open(combined_rttm_path, "w") as f:
    f.write("\n".join(rttm_lines_all))

print(f"✅ Converted {len(pairs)} CSV files to RTTM format")
print(f"   Combined RTTM saved to: {combined_rttm_path}")

# Show a sample to verify
print("\n📋 Sample RTTM lines (first 4):")
with open(combined_rttm_path) as f:
    for i, line in enumerate(f):
        if i >= 4: break
        print(f"   {line.strip()}")

In [ ]:
# ── Build pyannote Protocol ───────────────────────────────────────
# 
# A protocol is pyannote's way of saying:
#   "for training: use these files, for validation: use these files"
#
# We use 80% of files for training, 20% for validation.
# With ~20 training files, that's ~16 train, ~4 validation.

import random
random.seed(42)

# Shuffle and split
all_stems = [audio_path.stem for audio_path, _ in pairs]
random.shuffle(all_stems)

split_idx  = int(len(all_stems) * 0.8)
train_files = all_stems[:split_idx]
val_files   = all_stems[split_idx:]

print(f"Train files ({len(train_files)}): {train_files}")
print(f"Val files   ({len(val_files)}):   {val_files}")

# Write list files (pyannote needs these)
lists_dir = os.path.join(OUTPUT_DIR, "lists")
os.makedirs(lists_dir, exist_ok=True)

with open(os.path.join(lists_dir, "train.txt"), "w") as f:
    f.write("\n".join(train_files))

with open(os.path.join(lists_dir, "val.txt"), "w") as f:
    f.write("\n".join(val_files))

print("\n✅ Train/val split created")

In [ ]:
# ── Write database.yml ────────────────────────────────────────────
#
# This YAML file tells pyannote WHERE your audio files live.
# It maps file IDs to actual paths on disk.
#
# Format:
#   Databases:
#     BanglaData: /path/to/audio/{uri}.wav
#
# {uri} is a placeholder — pyannote replaces it with each file's stem name.

database_yml = f"""Databases:
  BanglaData: {TRAIN_AUDIO_DIR}/{{uri}}.wav

Protocols:
  BanglaData:
    SpeakerDiarization:
      Bangla:
        train:
          uri: {lists_dir}/train.txt
          annotation: {rttm_dir}/{{uri}}.rttm
        development:
          uri: {lists_dir}/val.txt
          annotation: {rttm_dir}/{{uri}}.rttm
"""

yml_path = os.path.join(OUTPUT_DIR, "database.yml")
with open(yml_path, "w") as f:
    f.write(database_yml)

print("✅ database.yml written")
print("\n📄 Content:")
print(database_yml)

In [ ]:
# ── Register the protocol with pyannote ──────────────────────────

from pyannote.database import registry, FileFinder

os.environ["PYANNOTE_DATABASE_CONFIG"] = yml_path
registry.load_database(yml_path)

# Get the protocol
protocol = registry.get_protocol(
    "BanglaData.SpeakerDiarization.Bangla",
    preprocessors={"audio": FileFinder()}
)

# Verify: iterate a few training files
print("✅ Protocol registered! Verifying training files...")
for i, file in enumerate(protocol.train()):
    print(f"  Train file {i+1}: {file['uri']}")
    print(f"    Audio:      {file['audio']}")
    print(f"    Annotation: {file['annotation']}")
    if i >= 2:
        print(f"  ... (showing first 3 only)")
        break

## Step 5: Load Pretrained Segmentation Model

### What model are we loading?
`pyannote/segmentation-3.0` — this is the segmentation backbone inside the
`speaker-diarization-community-1` pipeline you already use. We're pulling it out,
fine-tuning it, then putting it back.

It's a small transformer (~6MB) that processes 10-second audio chunks and outputs
frame-level speaker activity.

In [ ]:
import torch
from pyannote.audio import Model
from pyannote.audio.tasks import SpeakerDiarization as SpeakerDiarizationTask

# PyTorch 2.6+ security fix (same as your other notebooks)
# _original_torch_load = torch.load
# def _trusted_load(*args, **kwargs):
#     kwargs['weights_only'] = False
#     return _original_torch_load(*args, **kwargs)
# torch.load = _trusted_load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load the base segmentation model
print("\n📥 Loading pyannote/segmentation-3.0...")
model = Model.from_pretrained(
    "pyannote/segmentation-3.0",
     token=HF_TOKEN
)

print("✅ Segmentation model loaded")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Step 6: Set Up the Fine-tuning Task

### What is `SpeakerDiarization` task?
This is pyannote's training task wrapper. It tells the model:
- What data to use (our protocol)
- What loss function to use (powerset cross-entropy — the best one for this)
- How to generate training batches (random 10s chunks from your audio)

### Why `max_speakers_per_chunk=4`?
Your full recordings have up to 22 speakers, but in any given 10-second window,
you realistically have at most 3-4 active speakers. The model learns per-chunk,
not per-file.

In [ ]:
import torchaudio
from pyannote.core import Segment, Timeline

# ─── HELPER FUNCTION ─────────────────────────────────────────────
def add_annotated_region(file):
    """
    Calculates audio duration and sets the 'annotated' key 
    to cover the entire file.
    """
    # Get the audio path (already found by FileFinder)
    audio_path = file["audio"]
    
    # Get the duration of the audio file
    # (We use torchaudio.info for speed so we don't load the whole file)
    info = torchaudio.info(audio_path)
    duration = info.num_frames / info.sample_rate
    
    # Create a Timeline covering from 0 to duration
    # This tells the model: "Use the whole file for training"
    file["annotated"] = Timeline([Segment(0, duration)])
    
    return file

# ─── PATCH THE PROTOCOL ──────────────────────────────────────────
# We wrap the existing protocol generators to inject the 'annotated' key
# on the fly as files are loaded.

# 1. Save original methods (so we don't patch twice if you re-run)
if not hasattr(protocol, "_original_train"):
    protocol._original_train = protocol.train
    protocol._original_dev   = protocol.development

# 2. Define new generators
def train_gen():
    for f in protocol._original_train():
        yield add_annotated_region(f)

def dev_gen():
    for f in protocol._original_dev():
        yield add_annotated_region(f)

# 3. Apply the patch
protocol.train = train_gen
protocol.development = dev_gen

print("✅ Protocol patched! 'annotated' regions explicitly set to full duration.")

In [ ]:
# Create the fine-tuning task
task = SpeakerDiarizationTask(
    protocol,
    duration=CHUNK_DURATION,           # 10s chunks
    max_speakers_per_chunk=MAX_SPEAKERS_PER_CHUNK,  # 4 speakers max per chunk
    max_speakers_per_frame=1,          # at most 1 overlapping speakers at once
    batch_size=BATCH_SIZE,
    num_workers=2,                     # data loading workers
    loss="powerset",                   # best loss for diarization (from the 2023 paper)
)

# Attach the task to the model
model.task = task

print("✅ Fine-tuning task configured")
print(f"   Duration per chunk:        {CHUNK_DURATION}s")
print(f"   Max speakers per chunk:    {MAX_SPEAKERS_PER_CHUNK}")
print(f"   Batch size:                {BATCH_SIZE}")
print(f"   Loss:                      powerset cross-entropy")

## Step 9: Train! 🚀

### How long will this take?
With your dataset size and T4 GPU:
- Per epoch: ~5-10 minutes
- 5 epochs total: ~30-50 minutes
- Early stopping might cut it short if the model converges

### What should you watch?
- `train_loss` should go down each epoch
- `val_loss` should also go down (if it starts going up, early stopping will kick in)
- If `val_loss` diverges from `train_loss`, you're overfitting

### What is a `.ckpt` file?
A PyTorch Lightning checkpoint — it contains all the model weights,
optimizer state, and training metadata. It's everything needed to
continue training or run inference.

In [ ]:
import lightning.pytorch as pl  #using for pyannote.audio 4.x
from lightning.pytorch.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    RichProgressBar,
)
# import pytorch_lightning as pl  #using for pyannote.audio 3.3.1
# from pytorch_lightning.callbacks import (
#     ModelCheckpoint,
#     EarlyStopping,
#     RichProgressBar,
# )
from types import MethodType
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Configure optimizer (attach to model) ────────────────────────
def configure_optimizers(self):
    optimizer = Adam(self.parameters(), lr=LEARNING_RATE)
    
    # FIX 1: Removed 'verbose=True' (deprecated)
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )
    
    return {
        "optimizer": optimizer,
        "lr_scheduler": {
            "scheduler": scheduler,
            "monitor": "loss/val",   # FIX 2: Changed 'val_loss' to 'loss/val'
            "interval": "epoch",
        }
    }

# Bind the optimizer method to the model
model.configure_optimizers = MethodType(configure_optimizers, model)

# ── Callbacks ─────────────────────────────────────────────────────
checkpoint_callback = ModelCheckpoint(
    dirpath=OUTPUT_DIR,
    filename="best_model",
    monitor="loss/val",              # FIX 2: Changed 'val_loss' to 'loss/val'
    mode="min",
    save_top_k=1,
    verbose=True,
)

# early_stop_callback = EarlyStopping(
#     monitor="loss/val",              # FIX 2: Changed 'val_loss' to 'loss/val'
#     patience=3,
#     mode="min",
#     verbose=True,
# )

progress_callback = RichProgressBar()

# ── Build Trainer ─────────────────────────────────────────────────
torch.set_float32_matmul_precision("high")

trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    max_epochs=NUM_EPOCHS,
    callbacks=[
        checkpoint_callback,
        # early_stop_callback,
        progress_callback,
    ],
    log_every_n_steps=10,
    gradient_clip_val=0.5,
)

print("✅ Trainer created")
print(f"   Epochs:     up to {NUM_EPOCHS}")
print(f"   Callbacks:  ModelCheckpoint, EarlyStopping, RichProgressBar")

# ── Start training ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🚀 STARTING FINE-TUNING")
print("=" * 60)
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Best model: {OUTPUT_DIR}/best_model.ckpt")
print("=" * 60)

trainer.fit(model)

print("\n" + "=" * 60)
print("✅ FINE-TUNING COMPLETE")
print("=" * 60)

## Step 10: Verify the Saved Checkpoint

In [ ]:
import os

best_ckpt_path = os.path.join(OUTPUT_DIR, "best_model.ckpt")

if os.path.exists(best_ckpt_path):
    size_mb = os.path.getsize(best_ckpt_path) / (1024 * 1024)
    print(f"✅ Checkpoint saved successfully!")
    print(f"   Path: {best_ckpt_path}")
    print(f"   Size: {size_mb:.1f} MB")
    
    # Quick sanity check: load the checkpoint
    ckpt = torch.load(best_ckpt_path, map_location="cpu", weights_only=False)
    print(f"   Best val_loss: {ckpt.get('callbacks', {}).get('ModelCheckpoint', {}).get('best_model_score', 'N/A')}")
    print(f"   Epoch:         {ckpt.get('epoch', 'N/A')}")
else:
    # If best_model.ckpt doesn't exist, find whatever was saved
    print("⚠️  best_model.ckpt not found. Looking for other checkpoints...")
    ckpts = list(Path(OUTPUT_DIR).glob("*.ckpt"))
    if ckpts:
        best_ckpt_path = str(ckpts[-1])
        print(f"   Found: {best_ckpt_path}")
    else:
        print("❌ No checkpoints found at all. Check the training logs above.")

print(f"\n🎯 Use this path in your inference notebook:")
print(f"   FINETUNED_MODEL_PATH = '{best_ckpt_path}'")

## Step 11: Quick Sanity Check on a Training File

Let's run the fine-tuned segmentation model on one audio file and check that it produces
reasonable output. This does NOT replace your full inference notebook — it's just a smoke test.

In [ ]:
from pyannote.audio import Model, Inference
from pyannote.audio.pipelines import SpeakerDiarization as SpeakerDiarizationPipeline

# Load finetuned segmentation model from checkpoint
print("Loading finetuned segmentation model from checkpoint...")
finetuned_seg_model = Model.from_pretrained(
    best_ckpt_path,
    map_location=device
)
finetuned_seg_model.eval()
finetuned_seg_model.to(device)

print("✅ Model loaded from checkpoint")

# Run inference on the first training file as a quick test
test_audio = str(list(Path(TRAIN_AUDIO_DIR).glob("*.wav"))[0])
print(f"\n🎵 Testing on: {Path(test_audio).stem}")

# Use Inference to slide the model over the full audio
inference = Inference(finetuned_seg_model, window="sliding")
segmentation = inference(test_audio)

print(f"✅ Segmentation complete")
print(f"   Output shape: {segmentation.data.shape}  (frames x speaker_classes)")
print(f"   Duration:     {segmentation.sliding_window.duration:.1f}s chunks")
print("\n✅ Sanity check passed — model is working!")

## Step 12: How to Use This in Your Inference Notebook

### What to do after this notebook finishes:

**Step A: Save the checkpoint as a Kaggle Dataset**
1. Go to your Kaggle profile → Datasets → New Dataset
2. Upload `best_model.ckpt` from `/kaggle/working/finetuned_segmentation/`
3. Name it something like `bangla-finetuned-segmentation`

**Step B: Add it to your inference notebook**
1. Open your inference notebook
2. Go to "Data" tab on the right → "Add data" → find your dataset
3. It will be available at `/kaggle/input/bangla-finetuned-segmentation/best_model.ckpt`

**Step C: Modify your inference notebook (only 5 lines change)**

In your inference notebook, replace the pipeline loading section with:

```python
from pyannote.audio import Model, Pipeline

# Path to your finetuned checkpoint
FINETUNED_CKPT = "/kaggle/input/bangla-finetuned-segmentation/best_model.ckpt"

# Load the finetuned segmentation model
finetuned_seg_model = Model.from_pretrained(FINETUNED_CKPT, map_location=device)

# Load the full diarization pipeline (same as before)
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

# SWAP: replace the pipeline's segmentation model with your finetuned one
diarization_pipeline._segmentation.model = finetuned_seg_model
diarization_pipeline.to(device)

# Everything else in your inference notebook stays EXACTLY the same!
```

That's it. The embedding model and clustering stay the same. Only segmentation is swapped.

In [ ]:
# Print a ready-to-paste code snippet for your inference notebook
print("=" * 65)
print("📋 COPY-PASTE THIS INTO YOUR INFERENCE NOTEBOOK")
print("=" * 65)
snippet = '''
from pyannote.audio import Model, Pipeline

# 1. Path to your finetuned checkpoint (after uploading to Kaggle datasets)
FINETUNED_CKPT = "/kaggle/input/bangla-finetuned-segmentation/best_model.ckpt"

# 2. Load finetuned segmentation model
finetuned_seg_model = Model.from_pretrained(FINETUNED_CKPT, map_location=device)

# 3. Load the full pipeline (same as before)
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

# 4. Swap in the finetuned segmentation model
diarization_pipeline._segmentation.model = finetuned_seg_model
diarization_pipeline.to(device)

# 5. Instantiate with your tuned hyperparameters (same as before)
diarization_pipeline.instantiate({
    "clustering": {"threshold": CLUSTERING_THRESHOLD},
    "segmentation": {"min_duration_off": MIN_DURATION_OFF}
})

# Everything else stays the same!
'''
print(snippet)
print("=" * 65)

## 📝 Summary & Next Steps

### What just happened:
1. Your training CSVs were converted to RTTM format (pyannote's annotation format)
2. A pyannote protocol was created to feed your data to the model
3. The `pyannote/segmentation-3.0` model was fine-tuned on YOUR Bangla speech data
4. The best checkpoint (by validation loss) was saved to `/kaggle/working/finetuned_segmentation/best_model.ckpt`

### Expected improvement:
- Before fine-tune: DER ~0.32
- After fine-tune:  DER ~0.24–0.28 (expected, varies by data quality)

### If results disappoint:
- Try `NUM_EPOCHS = 10` (more training)
- Try `LEARNING_RATE = 5e-5` (slower, more careful fine-tuning)
- Make sure ALL training files were used (`len(pairs)` should be all your train files)

### If you get OOM (out of memory) errors:
- Reduce `BATCH_SIZE` from 8 to 4
- Reduce `MAX_SPEAKERS_PER_CHUNK` from 4 to 3

### Key files produced:
- `best_model.ckpt` → Your finetuned model weights. Upload to Kaggle datasets.
- `train.rttm`      → Combined annotation file. Keep for future re-training.